In [51]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import numpy as np
import json
import re

import sys
import os

# Ajout du dossier GNN au path pour permettre l'import direct de 'forecasting_models'
# Cela suppose que le notebook est dans le dossier 'Prediction' et que 'GNN' est un sous-dossier.
gnn_path = os.path.abspath('GNN')

if gnn_path not in sys.path:
    sys.path.insert(0, gnn_path)

print(f"Added to sys.path: {gnn_path}")

Added to sys.path: /home/caron/Bureau/ST-GNN-for-wildifre-prediction/Prediction/GNN


In [52]:

import datetime as dt
import numpy as np

def find_dates_between(start, end):
    start_date = dt.datetime.strptime(start, '%Y-%m-%d').date()
    end_date = dt.datetime.strptime(end, '%Y-%m-%d').date()

    delta = dt.timedelta(days=1)
    date = start_date
    res = []
    while date < end_date:
            res.append(date.strftime("%Y-%m-%d"))
            date += delta
    return res
    
def num_zone2_num_zone(df):
    dico = {}
    if 'departement' in df.columns:
         num_zones = np.sort(np.unique(df[df['departement'] == 6]['graph_id']))
         num_zone = [65,62,64,61,66,67,63]
         for i, gi in enumerate(num_zones):
             if i < len(num_zone):
                dico[gi] = num_zone[i]
    
    if 'graph_id' in df.columns:
        num_zones_all = np.unique(df['graph_id'])
        for gi in num_zones_all:
            if gi not in dico:
                dico[gi] = gi

    df['num_zone'] = df['graph_id'].map(dico)
    return df


In [53]:
import pickle

MODE = '06'

def _run_and_save_predictions(model, resolved_target: str):
    """
    Runs multi-horizon predictions and saves results in model.df_test.
    Follows logic from GNN.dataloader.test_dl_model.
    """
    from GNN.dataloader import filter_prediction
    from GNN.config import date_index, graph_id_index, scale_index, weight_index
    
    df_test = model.df_test
    target_col = TARGET_COLUMNS.get(resolved_target, resolved_target)
    
    # Run predictions with return_y=True for alignment
    pred_tensor, y_tensor = model.predict(df_test, return_y=True)
    
    # Run probabilities
    try:
        pred_proba_tensor, _ = model.predict_proba(df_test, return_y=True)
    except Exception:
        pred_proba_tensor = None

    # Convert to numpy
    if hasattr(y_tensor, 'detach'):
        y_tensor = y_tensor.detach().cpu().numpy()
    if hasattr(pred_tensor, 'detach'):
        pred_tensor = pred_tensor.detach().cpu().numpy()
    if pred_proba_tensor is not None and hasattr(pred_proba_tensor, 'detach'):
        pred_proba_tensor = pred_proba_tensor.detach().cpu().numpy()

    # Mock graphScale for filter_prediction
    class MockGS:
        def __init__(self):
            self.graph_method = graph_construct
            self.base = scale
            self.scale = scale

    # Align predictions
    pred_all, y_all, df_filtered = filter_prediction(MockGS(), df_test, pred_tensor, y_tensor, None, None)
    
    if pred_proba_tensor is not None:
        pred_proba_all, _, _ = filter_prediction(MockGS(), df_test, pred_proba_tensor, y_tensor, None, None)
    else:
        pred_proba_all = None

    horizon = model.horizon if hasattr(model, "horizon") else 0
    
    # Add columns for each horizon
    for H in range(horizon + 1):
        idx = -1 - (horizon - H)
        
        col_base = f"prediction_{target_col}_{H}"
        df_filtered[col_base] = pred_all[:, idx]
        
        if pred_proba_all is not None:
            for c in range(pred_proba_all.shape[-1]):
                df_filtered[f"{col_base}_C{c}"] = pred_proba_all[:, idx, c]
                
    full_df = pd.concat((model.df_train, model.df_val, model.df_test)).reset_index(drop=True)

    return df_filtered, model.df_train, full_df

clustering = "quantile"

clustering_suffix = f'{clustering}-5-Class-Dept'

TARGET_COLUMNS = {
    'nbsinister': f'nbsinister-{clustering_suffix}',
    'timeintervention': f'timeintervention-{clustering_suffix}',
    'DFE': 'DFE',
    'ressource': f'ressource-{clustering_suffix}',
    'burnedareaRoot': f'burnedareaRoot-{clustering_suffix}',
    'burnedarea': f'burnedarea-{clustering_suffix}',
}

if MODE == '06':
    import pickle
    
    scale = "3"
    graph_construct = "zonemeteo"
    horizon_train = '0'
    kdays = '10'
    under_sampling = "full"
    expe = "06"

    loss = 'ordinalNoCoverageWithGains-id{departement}'

    departement = '{departement}'
    
    fire_df, train_fire_df, full_fire_df = _run_and_save_predictions(pickle.load(open(f'GNN/firemen/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_{scale}_0_{graph_construct}_node/GRU_full_full_10_0_all_one_nbsinister_classification_{loss}/GRU_full_full_10_0_all_one_nbsinister_classification_{loss}.pkl', 'rb')), 'nbsinister')
    #res_df, train_res_df, full_res_df = _run_and_save_predictions(pickle.load(open(f'GNN/firemen/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_{scale}_0_{graph_construct}_node/GRU_full_full_10_0_all_one_ressource-quantile-5-Class-Dept_classification_flwki-id{departement}/GRU_full_full_10_0_all_one_ressource-quantile-5-Class-Dept_classification_flwki-id{departement}.pkl', 'rb')), 'ressource')
    #time_df, train_time_df, full_time_df = _run_and_save_predictions(pickle.load(open(f'GNN/firemen/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_{scale}_0_{graph_construct}_node/GRU_full_full_10_0_all_one_timeintervention-quantile-5-Class-Dept_classification_flwki-id{departement}/GRU_full_full_10_0_all_one_timeintervention-quantile-5-Class-Dept_classification_flwki-id{departement}.pkl', 'rb')), 'timeintervention')
    dfe_df, dfe_train, full_dfe_df = _run_and_save_predictions(pickle.load(open(f'GNN/firemen/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_{scale}_0_{graph_construct}_node/GRU_full_full_10_0_all_one_DFE_classification_wkloss/GRU_full_full_10_0_all_one_DFE_classification_wkloss.pkl', 'rb')), 'DFE')

elif MODE == 'bdiff':
    scale = "departement"
    expe = "default"
    graph_construct = "None"


    fire_df, train_fire_df, full_fire_df = _run_and_save_predictions(pickle.load(open(f'GNN/bdiff/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_{scale}_0_None_node/GRU_search_full_10_0_all_one_nbsinister-kmeans-5-Class-Dept_classification_flwk/GRU_search_full_10_0_all_one_nbsinister-kmeans-5-Class-Dept_classification_flwk.pkl', 'rb')), 'nbsinister')
    #res_df, train_res_df, full_res_df = _run_and_save_predictions(pickle.load(open(f'GNN/bdiff/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_departement_0_None_node/GRU_full_full_10_0_all_one_ressource-quantile-5-Class-Dept_classification_flwki-id{departement}/GRU_full_full_10_0_all_one_ressource-quantile-5-Class-Dept_classification_flwki-id{departement}.pkl', 'rb')), 'ressource')
    #res_df, train_res_df, full_res_df = _run_and_save_predictions(pickle.load(open(f'GNN/bdiff/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_departement_0_None_node/GRU_full_full_10_0_all_one_ressource-quantile-5-Class-Dept_classification_flwki-id{departement}/GRU_full_full_10_0_all_one_ressource-quantile-5-Class-Dept_classification_flwki-id{departement}.pkl', 'rb')), 'ressource')
    #time_df, train_time_df, full_time_df = _run_and_save_predictions(pickle.load(open(f'GNN/bdiff/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_departement_0_None_node/GRU_full_full_10_0_all_one_timeintervention-quantile-5-Class-Dept_classification_flwki-id{departement}/GRU_full_full_10_0_all_one_timeintervention-quantile-5-Class-Dept_classification_flwki-id{departement}.pkl', 'rb')), 'timeintervention')

elif MODE == 'firemen':
    scale = "departement"
    expe = "default"


    fire_df, train_fire_df, full_fire_df = _run_and_save_predictions(pickle.load(open(f'GNN/firemen/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_{scale}_0_None_node/GRU_full_full_10_0_all_one_nbsinister-kmeans-5-Class-Dept_classification_flwki-id{departement}/GRU_full_full_10_0_all_one_nbsinister-quantile-5-Class-Dept_classification_flwki-id{departement}.pkl', 'rb')), 'nbsinister')
    #res_df, train_res_df, full_res_df = _run_and_save_predictions(pickle.load(open(f'GNN/firemen/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_departement_0_None_node/GRU_full_full_10_0_all_one_ressource-quantile-5-Class-Dept_classification_flwki-id{departement}/GRU_full_full_10_0_all_one_ressource-quantile-5-Class-Dept_classification_flwki-id{departement}.pkl', 'rb')), 'ressource')
    #time_df, train_time_df, full_time_df = _run_and_save_predictions(pickle.load(open(f'GNN/firemen/firepoint/2x2/train/occurence_{expe}/check_z-score/full_all_departement_0_None_node/GRU_full_full_10_0_all_one_timeintervention-quantile-5-Class-Dept_classification_flwki-id{departement}/GRU_full_full_10_0_all_one_timeintervention-quantile-5-Class-Dept_classification_flwki-id{departement}.pkl', 'rb')), 'timeintervention')

id : departement
id : departement
id : departement
id : departement


In [54]:
######## A APPLIQUER A TOUT LES DATAFRAMES CHARGE (GRU + STATS_MODELS) ##########

# Process Dates for DFE
"""allDates_dfe = find_dates_between('2017-06-12', '2024-12-31')
full_dfe_df['date'] = full_dfe_df['date'].apply(lambda x : allDates_dfe[x] if x < len(allDates_dfe) else None)
full_dfe_df['date'] = pd.to_datetime(full_dfe_df['date'])
full_dfe_df['num_zone'] = full_dfe_df['num_zone']

full_dfe_df = full_dfe_df[full_dfe_df['date'] >= '2018-01-01']"""

# Process Dates for Fire/Time/Res
allDates_fire = find_dates_between('2017-06-12', '2024-12-31')

dfs = [
    "full_fire_df",
    #"full_dfe_df",
    #"train_fire_df",
    "full_dfe_df"
]

for name in dfs:
    df = globals().get(name, None)
    
    if df is None:
        continue

    # sécurise aussi si colonne absente
    if 'date' in df.columns:
        df['date'] = df['date'].apply(
            lambda x: allDates_fire[x] if pd.notna(x) and x < len(allDates_fire) else None
        )
        df['date'] = pd.to_datetime(df['date'])

    df = num_zone2_num_zone(df)

    # remet à jour la variable originale
    globals()[name] = df

In [55]:
full_fire_df[full_fire_df['date'] >= '2024-01-12']['DFE'].unique()

array([nan,  1.,  0.,  2.,  3.])

In [56]:
if MODE == '06':
    fire_df = fire_df[fire_df['departement'] == 6]

In [57]:
KEYS = ["date", "num_zone"] ######" NE CHANGE PAS ##########

#columns = ['nbsinister', 'ressource', 'time_intervention',  f'nbsinister-{clustering_suffix}', f'timeintervention-{clustering_suffix}', f'ressource-{clustering_suffix}']

#train_fire_df_key = full_fire_df[KEYS + columns]

#full_dfe_df = full_dfe_df.merge(train_fire_df_key, on=KEYS, how='left')

In [58]:

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# Spline FE: mu(0..4) + passages (avec 4)
# -----------------------------
LEVELS = [0, 1, 2, 3, 4]

PASSAGES = {
    1: [(0,1), (1,2), (2,3), (3,4)],
    2: [(0,2), (1,3), (2,4)],
    3: [(0,3), (1,4)],
    4: [(0,4)],
}

# Redefine compute_score_for_k to match comparison_analysis logic
# 1. Use Median instead of Mean for average delta
# 2. Use Weights: w_min=0.5, w_viol=0.5 (defaults in comparison_analysis)

def fit_spline_mu(df, df_spline=5):
    """
    Fits a spline model Y ~ bs(score) + C(zone) + C(date)
    and returns the predicted mean for each level in LEVELS.
    """
    # Hardcoded LEVELS for safety, matching notebook usage
    LEVELS = [0, 1, 2, 3, 4]

    d = df.dropna(subset=["score", "Y", "zone", "date"]).copy()
    d["score"] = d["score"].clip(0, 4)
    
    # Ensure categorical types for fixed effects
    d["zone"] = d["zone"].astype("category")
    d["date"] = d["date"].astype("category")

    formula = (
        f"Y ~ bs(score, df={df_spline}, degree=3, include_intercept=False, "
        f"lower_bound=0, upper_bound=4) + C(zone) + C(date)"
    )
    
    try:
        fit = smf.ols(formula, data=d).fit(cov_type="HC1")
    except Exception as e:
        # Fallback if too few data points or other issues
        print(f"Warning: Spline fit failed: {e}")
        return {lvl: np.nan for lvl in LEVELS}, None

    template = d[["zone", "date"]].copy()
    mu = {}
    for s in LEVELS:
        tmp = template.copy()
        tmp["score"] = s
        pred = fit.predict(tmp)
        mu[s] = pred.mean()
        
    return mu, fit

def compute_score_for_k(mu, sigma, k, lvl_counts,
                        min_n=1, min_k=1,
                        w_avg=1.0, w_min=1.0, w_neg=1.0, w_viol=1.0,  # FIXED WEIGHTS
                        min_gain=0.0):
    """
    Computes the score for a given k based on deltas between levels.
    Returns score and coverage (number of valid pairs |P_k*|).
    """
    pairs = PASSAGES.get(k, [])
    deltas = []
    
    coverage = 0

    for (a, b) in pairs:
        # Filter based on min_n
        n_a = lvl_counts.get(a, 0)
        n_b = lvl_counts.get(b, 0)
        
        if (n_a >= min_n) and (n_b >= min_n):
            delta = mu[b] - (mu[a] + min_gain)
            deltas.append(delta)
            coverage += min(n_a, n_b)
    
    # Filter based on min_k
    if coverage < min_k:
        return np.nan, coverage

    if len(deltas) == 0:
        return np.nan, coverage

    deltas = np.array(deltas)
    # Standardize by sigma
    deltas_std = deltas / sigma
    
    # FIXED: Use MEDIAN for average, consistent with comparison_analysis
    avg_delta_std = np.median(deltas_std) 
    
    min_delta_std = np.min(deltas_std)
    neg_mass_std = np.mean(np.clip(-deltas_std, 0.0, None))
    viol_rate = np.mean(deltas_std < 0.0)
    
    score = (
        (w_avg * avg_delta_std
        + w_min * min_delta_std) / 2
        - w_neg * neg_mass_std * (1 + w_viol * viol_rate)
    )
    
    # Coverage weighting (disabled as per comparison_analysis logic which doesn't seem to force it here)
    score_adj = score
    return score_adj, coverage

# Redefine evaluation_scoring to capture new defaults
def evaluation_scoring(ypred, ytrue, dates, zones, df_spline=5, min_n=5, min_k=0, min_gain=[0.0, 0.0, 0.0, 0.0]):
    """
    Main function to evaluate monotonic comparison analysis.
    Returns score_high, score_low, coverage_k.
    """
    df = pd.DataFrame({
        "score": ypred,
        "Y": ytrue,
        "date": dates,
        "zone": zones
    })
    
    # Calculate level counts
    df["_lvl"] = df["score"].clip(0, 4).astype(int)
    lvl_counts = df["_lvl"].value_counts().to_dict()
    
    sigma = df["Y"].std()
    if sigma == 0 or np.isnan(sigma):
        sigma = 1.0 # Avoid division by zero
        
    mu, fit = fit_spline_mu(df, df_spline=df_spline)
    
    if all(np.isnan(list(mu.values()))):
        return np.nan, np.nan, {}
    
    score_adj_k = {}
    coverage_k = {}
    
    # Using defaults from re-defined compute_score_for_k (w_min=0.5, w_viol=0.5)
    for k in [1, 2, 3, 4]:
        # Handle min_gain being float (scalar) or list
        if isinstance(min_gain, (int, float)):
             min_g = min_gain
        else:
             min_g = min_gain[k - 1]
             
        score, coverage = compute_score_for_k(mu, sigma, k, lvl_counts, min_n=min_n, min_k=min_k, min_gain=min_g)
        score_adj_k[k] = score
        coverage_k[k] = coverage
        
    score_low = score_adj_k[1] + score_adj_k[2]
    score_high = score_adj_k[3] + score_adj_k[4]
    
    if np.isnan(score_low):
        score_low = 0.0
    if np.isnan(score_high):
        score_high = 0.0
        
    res = {
        "score_k1": score_adj_k[1],
        "score_k2": score_adj_k[2],
        "score_k3": score_adj_k[3],
        "score_k4": score_adj_k[4],
        "score_low": score_low,
        "score_high": score_high,
        "coverage": coverage_k
    }
    
    return res, fit


In [59]:

TARGET_COLUMNS = {
    'nbsinister': f'nbsinister-{clustering_suffix}',
    'time_intervention': f'timeintervention-{clustering_suffix}',
    'DFE': 'DFE',
    'ressource': f'ressource-{clustering_suffix}',
    'burnedareaRoot': f'burnedareaRoot-{clustering_suffix}',
    'burnedarea': f'burnedarea-{clustering_suffix}',
}

min_n = 1
min_k = 1
fire_min_gain = [0.0, 0.0, 0.0, 0.0]
res_min_gain = [0.0, 0.0, 0.0, 0.0]
time_min_gain = [0.0, 0.0, 0.0, 0.0]

res_dfe = {}
fit_dfe = {}

for (target, y_col) in TARGET_COLUMNS.items():
    if 'full_dfe_df' in globals():
        res_dfe[target], fit_dfe[target] = evaluation_scoring(full_dfe_df[f'DFE'], full_dfe_df[target], full_dfe_df['date'], full_dfe_df['num_zone'], df_spline=5, min_n=min_n, min_gain=fire_min_gain)
    
    """if target == 'nbsinister':
        res_fire, fit_fire = evaluation_scoring(fire_df[f'prediction_{y_col}_0'], fire_df[y_col], fire_df['date'], fire_df['num_zone'], df_spline=5, min_n=min_n, min_gain=fire_min_gain)
    elif target == 'ressource':
        res_res, fit_res = evaluation_scoring(res_df[f'prediction_{y_col}_0'], res_df[y_col], res_df['date'], res_df['num_zone'], df_spline=5, min_n=min_n, min_gain=res_min_gain)
    elif target == 'time_intervention':
        res_time, fit_time = evaluation_scoring(time_df[f'prediction_{y_col}_0'], time_df[y_col], time_df['date'], time_df['num_zone'], df_spline=5, min_n=min_n, min_gain=time_min_gain)"""

In [60]:
PLOT = False
from pathlib import Path

if PLOT:

    # Plot fitted spline for each experiment (score in [0,4])
    # This cell plots the marginal fitted mean (averaged over zone/date) for scores in [0,4]
    import numpy as np
    import matplotlib.pyplot as plt

    targets = ['nbsinister', 'ressource', 'time_intervention', 'DFE']
    # prediction fits from evaluation_scoring
    pred_fits = {
        'nbsinister': globals().get('fit_fire', None),
        'ressource': globals().get('fit_res', None),
        'time_intervention': globals().get('fit_time', None),
    }

    xs = np.linspace(0, 4, 200)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()

    for i, target in enumerate(targets):
        ax = axes[i]

        # DFE fit (trained on dfe_train)
        fit_d = None
        try:
            fit_d = fit_dfe.get(target)
        except Exception:
            fit_d = None

        # Prediction fit (fire/res/time)
        fit_p = pred_fits.get(target)

        plotted_any = False

        def mean_pred_over_template(fit_obj, xs):
            # build a template from the data used to fit (zone/date) and return mean predictions
            try:
                df_template = fit_obj.model.data.frame[['zone', 'date']].drop_duplicates().copy()
            except Exception:
                # fallback: try to get unique zone/date from the global dfs if available
                if target == 'DFE' and 'dfe_train' in globals():
                    df_template = dfe_train[['num_zone', 'date']].rename(columns={'num_zone':'zone'}).drop_duplicates().copy()
                    df_template['zone'] = df_template['zone'].astype('category')
                elif target in ['nbsinister', 'ressource', 'time_intervention']:
                    dfname = {'nbsinister': 'fire_df', 'ressource': 'res_df', 'time_intervention': 'time_df'}[target]
                    if dfname in globals():
                        tmpdf = globals()[dfname]
                        df_template = tmpdf[['num_zone', 'date']].rename(columns={'num_zone':'zone'}).drop_duplicates().copy()
                        df_template['zone'] = df_template['zone'].astype('category')
                    else:
                        return None
            preds = []
            for x in xs:
                tmp = df_template.copy()
                tmp['score'] = float(x)
                try:
                    preds.append(fit_obj.predict(tmp).mean())
                except Exception:
                    return None
            return np.array(preds)

        if fit_d is not None:
            preds_d = mean_pred_over_template(fit_d, xs)
            if preds_d is not None:
                ax.plot(xs, preds_d, label='DFE fit', color='C0')
                plotted_any = True

        if fit_p is not None:
            preds_p = mean_pred_over_template(fit_p, xs)
            if preds_p is not None:
                ax.plot(xs, preds_p, label='prediction fit', color='C1', linestyle='--')
                plotted_any = True

        if not plotted_any:
            ax.text(0.5, 0.5, 'No fit available', ha='center', va='center')

        ax.set_title(target)
        ax.set_xlabel('score')
        ax.set_ylabel('predicted Y')
        ax.legend()

    plt.tight_layout()
    plt.show()

In [61]:
full_res_df = full_fire_df.copy(deep=True)
full_time_df = full_fire_df.copy(deep=True)

In [62]:

# --- FWI Scoring Analysis ---
print("\n" + "="*40)
print("Starting FWI Scoring Analysis (Quantile Clustered)")
print("="*40)

# Define Quantiles for Clustering
# User request: "5 classes (0 - 4) : [0.5, 0.75, 0.95]"
# Interpreted as:
# Class 0: [0, 0.5]
# Class 1: (0.5, 0.75]
# Class 2: (0.75, 0.95]
# Class 3: (0.95, 0.99]  <-- Inferred split
# Class 4: (0.99, 1.0]   <-- Inferred split
#
# YOU CAN MODIFY THESE QUANTILES EASILY HERE:
FWI_QUANTILES = [0.5, 0.75, 0.95, 0.99] 

print(f"Using FWI Quantiles: {FWI_QUANTILES}")

# Define target mapping: Name -> (TargetCol, DataFrame)
target_map = {
    "Fire": ("nbsinister", 'full_fire_df'),
    "Ressource": ("ressource", 'full_fire_df'),
    "Time": ("time_intervention", 'full_fire_df'),
    "burnedareaRoot": ("burnedareaRoot", 'full_fire_df'),
    "burnedarea": ("burnedarea", 'full_fire_df'),
}

# Helper to discretize FWI
def discretize_fwi(series, quantiles):
    # Calculate actual values for quantiles
    bins = [series.min() - 0.001] + series.quantile(quantiles).tolist() + [series.max() + 0.001]
    # Use cut to assign labels 0 to N
    labels = range(len(bins) - 1)
    return pd.cut(series, bins=bins, labels=labels, include_lowest=True).astype(int)

for target_name, (target_col, df_var_name) in target_map.items():
    print(f"\nProcessing {target_name}...")
    
    if df_var_name not in locals():
        print(f"Skipping {target_name}: {df_var_name} not loaded.")
        continue
    
    gt_df = locals()[df_var_name].copy()
    
    if 'fwi_mean' not in gt_df.columns:
        print(f"Warning: 'fwi_mean' not found in {df_var_name}. Skipping.")
        continue
        
    # Discretize FWI
    gt_df['fwi_score'] = discretize_fwi(gt_df['fwi_mean'], FWI_QUANTILES)
    
    print(f"FWI Discretization Distribution for {target_name}:")
    print(gt_df['fwi_score'].value_counts().sort_index())
    
    results = []
    
    # Global Scoring
    res_global, _ = evaluation_scoring(
        gt_df['fwi_score'], 
        gt_df[target_col], 
        gt_df['date'], 
        gt_df['num_zone'], 
        df_spline=5, 
        min_n=1,
        min_gain=0.0
    )
    
    if isinstance(res_global, tuple):
        res_global = res_global[0] if len(res_global) != 3 else {}
        
    if isinstance(res_global, dict) and 'score_k1' in res_global:
        cov = res_global.get('coverage', {})
        results.append({
            'Dataset': 'Global',
            'k1': res_global.get('score_k1'),
            'k2': res_global.get('score_k2'),
            'k3': res_global.get('score_k3'),
            'k4': res_global.get('score_k4'),
            'cov_k1': cov.get(1, 0),
            'cov_k2': cov.get(2, 0),
            'cov_k3': cov.get(3, 0),
            'cov_k4': cov.get(4, 0),
            'cov_total': sum(cov.values())
        })
        
    # Yearly Scoring
    if pd.api.types.is_datetime64_any_dtype(gt_df['date']):
        years = sorted(gt_df['date'].dt.year.unique())
        for year in years:
            df_year = gt_df[gt_df['date'].dt.year == year]
            if len(df_year) == 0: continue
            
            res_year = evaluation_scoring(
                df_year['fwi_score'], 
                df_year[target_col], 
                df_year['date'], 
                df_year['num_zone'], 
                df_spline=5, 
                min_n=1,
                min_gain=0.0
            )
            
            if isinstance(res_year, tuple):
                res_year = res_year[0] if len(res_year) != 3 else {}

            if isinstance(res_year, dict) and 'score_k1' in res_year:
                cov = res_year.get('coverage', {})
                results.append({
                    'Dataset': str(year),
                    'k1': res_year.get('score_k1'),
                    'k2': res_year.get('score_k2'),
                    'k3': res_year.get('score_k3'),
                    'k4': res_year.get('score_k4'),
                    'cov_k1': cov.get(1, 0),
                    'cov_k2': cov.get(2, 0),
                    'cov_k3': cov.get(3, 0),
                    'cov_k4': cov.get(4, 0),
                    'cov_total': sum(cov.values())
                })

    if results:
        df_res = pd.DataFrame(results)
        print(f"\nFWI Results for {target_name}:")
        print(df_res)
        
        csv_filename = f"fwi_scores_{target_name}.csv"
        df_res.to_csv(Path(MODE) / csv_filename, index=False)
        print(f"Saved FWI scores to {csv_filename}")



Starting FWI Scoring Analysis (Quantile Clustered)
Using FWI Quantiles: [0.5, 0.75, 0.95, 0.99]

Processing Fire...
FWI Discretization Distribution for Fire:
fwi_score
0    9793
1    4896
2    3917
3     784
4     196
Name: count, dtype: int64

FWI Results for Fire:
  Dataset        k1        k2        k3        k4  cov_k1  cov_k2  cov_k3  \
0  Global  0.024117  0.048235  0.149076  0.367402    9793    4897     980   
1    2017  0.000000  0.000000  0.000000  0.000000     811     538     245   
2    2018 -0.940074 -0.915833 -1.487064 -2.772121     902     304      41   
3    2019 -0.022390 -0.013513  0.287676  0.678118    1261     720      85   
4    2020 -0.205020 -0.374889 -0.222992  0.487196    1392     680     139   
5    2021 -0.086365  0.024638  0.097512  0.049277    1218     587     107   
6    2022  0.022233  0.051979  0.234125  0.571342    1527     772     169   
7    2023  0.048278  0.127452  0.243727  0.526890    1407     610     157   
8    2024 -0.394819 -0.569174 -1.140081

In [63]:
if MODE == '06':
    # --- Multi-Target DFE Scoring (With Strict Merging) ---
    print("Starting Multi-Target DFE Scoring Analysis (Aligned with Comparison Analysis)...")

    # Define target mapping: Name -> (TargetCol, DataFrame)
    # Note: We assume full_fire_df, full_res_df, full_time_df are available now.
    # Define target mapping: Name -> (TargetCol, DataFrame)
    # Note: Using RAW columns for Ground Truth (Y), not clustered ones.
    target_map = {
        "Fire": ("nbsinister", 'full_fire_df'),
        "Ressource": ("ressource", 'full_res_df'),
        "Time": ("time_intervention", 'full_time_df') 
    }

    # Identify prediction column from DFE
    # User requested "DFE" column explicitly (The official index)
    pred_col = 'DFE'
    if pred_col not in full_dfe_df.columns:
        print(f"WARNING: {pred_col} not found in full_dfe_df! Falling back to 'prediction_DFE_0' if available.")
        if 'prediction_DFE_0' in full_dfe_df.columns:
            pred_col = 'prediction_DFE_0'
        else:
            print("ERROR: Neither 'DFE' nor 'prediction_DFE_0' found.")

    # Ensure parameters are defined
    if 'min_n' not in locals(): min_n = 1
    if 'fire_min_gain' not in locals(): fire_min_gain = 0.0

    keys = ['date', 'num_zone']
    # Prepare DFE key dataframe for merging
    dfe_key = full_dfe_df[keys + [pred_col]].copy()

    # Ensure DFE date is datetime
    if not pd.api.types.is_datetime64_any_dtype(dfe_key['date']):
        dfe_key['date'] = pd.to_datetime(dfe_key['date'])

    for target_name, (target_col, df_var_name) in target_map.items():
        print(f"\n{'='*20}\nEvaluation: DFE Prediction vs {target_name}\nTarget Column: {target_col}\nDataFrame: {df_var_name}\n{'='*20}")
        
        # Retrieve the Ground Truth DataFrame
        if df_var_name not in locals():
            print(f"Skipping {target_name}: {df_var_name} not loaded.")
            continue
        
        gt_df = locals()[df_var_name]
        
        if target_col not in gt_df.columns:
            print(f"Skipping {target_name}: Column {target_col} not found in {df_var_name}.")
            continue

        # Ensure GT date is datetime
        if not pd.api.types.is_datetime64_any_dtype(gt_df['date']):
            gt_df['date'] = pd.to_datetime(gt_df['date'])

        # --- MERGING TO ALIGN PREDICTION WITH GROUND TRUTH ---
        # Comparison Analysis uses: fire_df_filter.merge(dfe_key, on=KEYS, how="left")
        # This keeps only rows present in the Ground Truth (filtering out days where GT is missing but DFE exists)
        
        # Rename prediction column in key to avoid collision if GT also has it (e.g. DFE vs DFE_x)
        safe_pred_col = "dfe_score_pred_final"
        dfe_key_renamed = dfe_key.rename(columns={pred_col: safe_pred_col})
        
        merged_df = gt_df.merge(dfe_key_renamed, on=keys, how='left')
        
        print(f"Merged {len(gt_df)} rows from GT with DFE. Result: {len(merged_df)} rows.")
        
        # Check for NaNs in prediction after merge
        missing_preds = merged_df[safe_pred_col].isna().sum()
        if missing_preds > 0:
            print(f"Warning: {missing_preds} rows have missing DFE predictions after merge. Filling with 0 or dropping?")
            merged_df = merged_df.dropna(subset=[safe_pred_col])
            print(f"Dropped rows with missing predictions. New length: {len(merged_df)}")

        results = []
        
        # Global Scoring on Merged Data

        
        res_global, _ = evaluation_scoring(
            merged_df[safe_pred_col], 
            merged_df[target_col], 
            merged_df['date'], 
            merged_df['num_zone'], 
            df_spline=5, 
            min_n=min_n, 
            min_gain=fire_min_gain
        )
        
        # Handle Global Results
        if isinstance(res_global, tuple):
            res_global = res_global[0] if len(res_global) != 3 else {}
            
        if isinstance(res_global, dict) and 'score_k1' in res_global:
            cov = res_global.get('coverage', {})
            results.append({
                'Dataset': 'Global',
                'k1': res_global.get('score_k1'),
                'k2': res_global.get('score_k2'),
                'k3': res_global.get('score_k3'),
                'k4': res_global.get('score_k4'),
                'cov_k1': cov.get(1, 0),
                'cov_k2': cov.get(2, 0),
                'cov_k3': cov.get(3, 0),
                'cov_k4': cov.get(4, 0),
                'cov_total': sum(cov.values())
            })
        else:
            print("Error in Global scoring result format or Global evaluation failed.")

        # Yearly Scoring
        years = sorted(merged_df['date'].dt.year.unique())
        for year in years:
            df_year = merged_df[merged_df['date'].dt.year == year]
            if len(df_year) == 0: continue
            
            res_year = evaluation_scoring(
                df_year[safe_pred_col], 
                df_year[target_col], 
                df_year['date'], 
                df_year['num_zone'], 
                df_spline=5, 
                min_n=min_n, 
                min_gain=fire_min_gain
            )
            
            if isinstance(res_year, tuple):
                res_year = res_year[0] if len(res_year) != 3 else {}

            if not isinstance(res_year, dict) or 'score_k1' not in res_year:
                continue
            
            cov = res_year.get('coverage', {})
            results.append({
                'Dataset': str(year),
                'k1': res_year.get('score_k1'),
                'k2': res_year.get('score_k2'),
                'k3': res_year.get('score_k3'),
                'k4': res_year.get('score_k4'),
                'cov_k1': cov.get(1, 0),
                'cov_k2': cov.get(2, 0),
                'cov_k3': cov.get(3, 0),
                'cov_k4': cov.get(4, 0),
                'cov_total': sum(cov.values())
            })
        
        # Output Table
        if results:
            df_res = pd.DataFrame(results)
            print(f"\nResults for {target_name}:")
            print(df_res)
            csv_filename = f"dfe_scores_{target_name}.csv"
            df_res.to_csv(Path(MODE) / csv_filename, index=False)
            print(f"Saved scores to {csv_filename}")
            print(f"\nLaTeX Table ({target_name}):")
            print(df_res.to_latex(index=False, float_format="%.3f"))
        else:
            print("No results generated.")

Starting Multi-Target DFE Scoring Analysis (Aligned with Comparison Analysis)...

Evaluation: DFE Prediction vs Fire
Target Column: nbsinister
DataFrame: full_fire_df
Merged 19586 rows from GT with DFE. Result: 19586 rows.
Dropped rows with missing predictions. New length: 5544

Results for Fire:
  Dataset        k1        k2        k3        k4  cov_k1  cov_k2  cov_k3  \
0  Global  0.050962  0.159109  0.268219  0.318218    3009    1600     522   
1    2017  0.000000  0.000000  0.000000  0.000000     386     180      61   
2    2018 -1.401702 -1.493397 -2.655163       NaN     284      56       1   
3    2019  0.024870  0.134560  0.438082       NaN     343     117       8   
4    2020 -0.564503  0.121759  0.334672  0.243519     198      55       9   
5    2021 -0.522966 -0.061504  0.017971  0.028544     298     145      38   
6    2022 -0.202471 -0.299556  0.201626  1.036028     448     187      35   
7    2023 -0.049878  0.007817  0.236055  0.341683     420     225      76   
8    2024

In [64]:
full_dfe_df['date']

0      2017-06-21
1      2017-06-21
2      2017-06-21
3      2017-06-21
4      2017-06-21
          ...    
5539   2024-09-27
5540   2024-09-27
5541   2024-09-27
5542   2024-09-27
5543   2024-09-27
Name: date, Length: 5544, dtype: datetime64[ns]